In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "notebooks").is_dir() else Path.cwd().parent

protein_path = ROOT / "data/processed/human_proteins_with_string.parquet"
string_path = ROOT / "data/raw/string/9606.protein.links.detailed.v12.5.txt.gz"

protein_df = pd.read_parquet(protein_path)

print("Canonical proteins:", len(protein_df))
print("STRING edge file exists:", string_path.exists())

Canonical proteins: 18841
STRING edge file exists: True


In [2]:
# Sirf first 10 PPI rows dekho; full large file abhi RAM mein load nahi karni
ppi_sample = pd.read_csv(
    string_path,
    sep=r"\s+",
    compression="gzip",
    nrows=10
)

ppi_sample

,protein1,protein2,neighborhood,fusion,cooccurence,coexpression,experimental,database,textmining,combined_score
0,9606.ENSP00000000233,9606.ENSP00000000412,0,0,0,97,0,0,151,200
1,9606.ENSP00000000233,9606.ENSP00000037243,0,0,0,159,0,0,63,178
2,9606.ENSP00000000233,9606.ENSP00000040663,0,0,0,83,191,0,0,226
3,9606.ENSP00000000233,9606.ENSP00000046087,0,0,0,0,0,0,158,158
4,9606.ENSP00000000233,9606.ENSP00000054666,0,0,0,88,44,0,383,415
5,9606.ENSP00000000233,9606.ENSP00000156084,0,0,0,0,0,0,153,153
6,9606.ENSP00000000233,9606.ENSP00000158762,0,0,0,63,46,0,835,839
7,9606.ENSP00000000233,9606.ENSP00000160827,0,0,0,72,45,500,0,518
8,9606.ENSP00000000233,9606.ENSP00000163416,0,0,0,60,65,0,212,246
9,9606.ENSP00000000233,9606.ENSP00000168216,0,0,0,219,0,0,0,219


In [3]:
print("PPI columns:")
print(ppi_sample.columns.tolist())

print("\nCombined score examples:")
print(ppi_sample["combined_score"].tolist())

PPI columns:
['protein1', 'protein2', 'neighborhood', 'fusion', 'cooccurence', 'coexpression', 'experimental', 'database', 'textmining', 'combined_score']

Combined score examples:
[200, 178, 226, 158, 415, 153, 839, 518, 246, 219]


In [4]:
from tqdm.auto import tqdm

valid_string_ids = set(protein_df["string_id"])

thresholds = [400, 700, 900]

total_raw_edges = 0
edges_with_both_proteins_mapped = 0
threshold_counts = {threshold: 0 for threshold in thresholds}

chunk_reader = pd.read_csv(
    string_path,
    sep=r"\s+",
    compression="gzip",
    usecols=["protein1", "protein2", "combined_score"],
    chunksize=250_000,
)

for chunk in tqdm(chunk_reader, desc="Scanning STRING PPI edges"):
    total_raw_edges += len(chunk)

    both_proteins_present = (
        chunk["protein1"].isin(valid_string_ids)
        & chunk["protein2"].isin(valid_string_ids)
    )

    mapped_edges = chunk.loc[both_proteins_present]
    edges_with_both_proteins_mapped += len(mapped_edges)

    for threshold in thresholds:
        threshold_counts[threshold] += (
            mapped_edges["combined_score"] >= threshold
        ).sum()

summary_df = pd.DataFrame(
    {
        "combined_score_threshold": thresholds,
        "retained_edges": [threshold_counts[t] for t in thresholds],
    }
)

summary_df["percentage_of_mapped_edges"] = (
    summary_df["retained_edges"]
    / edges_with_both_proteins_mapped
    * 100
).round(2)

print("Total raw STRING edges:", f"{total_raw_edges:,}")
print("Edges with both proteins in our table:", f"{edges_with_both_proteins_mapped:,}")

summary_df

c:\Users\rajdeep\Desktop\Capstone Project_GNN\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Scanning STRING PPI edges: 44it [00:39,  1.11it/s]

Total raw STRING edges: 10,826,054
Edges with both proteins in our table: 10,448,006


,combined_score_threshold,retained_edges,percentage_of_mapped_edges
0,400,2082528,19.93
1,700,533818,5.11
2,900,236840,2.27
